# Forecast

## Import

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score
from pathlib import Path

## Data

In [3]:
base_ordner = Path().resolve().parent
proc_data_ordner = base_ordner / "data" / "processed"

df_l = pd.read_csv(proc_data_ordner / "lauschke_kpis.csv")

In [6]:
x = df_l["Year"].values.reshape(-1, 1)
y = df_l["CarbonEmissions"].values

baseline_co2 = df_l[df_l["Year"] == 2015]["CarbonEmissions"].values[0]
target_2030 = baseline_co2 * 0.4

print(f"Baseline 2015: {baseline_co2:.3f} t")
print(f"Ziel 2030 (−60%): {target_2030:.3f} t")

Baseline 2015: 569254.800 t
Ziel 2030 (−60%): 227701.920 t


## First Models

In [7]:
#linear
linear_model = LinearRegression()
linear_model.fit(x, y)

r2_lin = r2_score(y, linear_model.predict(x))
r2_lin

0.8610365713910719

In [8]:
#poly
poly_model = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
poly_model.fit(x, y)

r2_poly = r2_score(y, poly_model.predict(x))
r2_poly

0.9638151213328152

## Forecast

Polynomial model chosen

In [9]:
years_future = np.array([2026, 2027, 2028, 2029, 2030]).reshape(-1, 1)

forecast_poly = poly_model.predict(years_future)

df_forecast = pd.DataFrame({
    "Year" : [2026, 2027, 2028, 2029, 2030],
    "CO2_poly" : forecast_poly.round(0),
})

df_forecast["poly vs 2015"] = ((df_forecast["CO2_poly"] - baseline_co2) / baseline_co2 * 100).round(2)

print(f"Ziel 2030: {target_2030}")
df_forecast

Ziel 2030: 227701.92000000004


,Year,CO2_poly,poly vs 2015
0,2026,304770.0,-46.46
1,2027,323342.0,-43.20
2,2028,349470.0,-38.61
3,2029,383153.0,-32.69
4,2030,424391.0,-25.45


### Q4

The polynomial Model suggests that give its current trjectory, Lauschke GmbH will miss its hypothetical 2030 climate target.

In [10]:
df_forecast.to_csv(proc_data_ordner / "forecast_2030.csv", index=False)
print("done")

done
